In [1]:
# Installs to get OPUS-MT
!pip install transformers sacrebleu sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.8 MB/s eta 0:00:00


In [4]:
# Mount drive to retrieve datasets
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
# Parse SGM files, using re import, so it is interpretable for model
import re
import sacrebleu
from tqdm import tqdm # Using tqdm to track eval progress

def parse_sgm(filepath):
    sentences = []
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line.startswith('<seg'):
                match = re.search(r'<seg[^>]*>(.*?)</seg>', line)
                if match:
                    sentences.append(match.group(1).strip())
    return sentences

fr_test_path = '/content/drive/MyDrive/cs505_data/newstest2014-fren-src.fr.sgm'
en_test_path = '/content/drive/MyDrive/cs505_data/newstest2014-fren-ref.en.sgm'

test_fr = parse_sgm(fr_test_path)
test_en = parse_sgm(en_test_path)

print(f"FR: {test_fr[0]}")
print(f"EN: {test_en[0]}")



Test sentences: 3003
FR: L'affaire NSA souligne l'absence totale de débat sur le renseignement
EN: NSA Affair Emphasizes Complete Lack of Debate on Intelligence


In [6]:
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-fr-en"
model = MarianMTModel.from_pretrained(model_name)
tokenizer = MarianTokenizer.from_pretrained(model_name)
print("Model loaded successfully")

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

Model loaded successfully


In [7]:
# BLEU Eval
def translate_opus(sentences, batch_size=32):
    translations = []
    for i in tqdm(range(0, len(sentences), batch_size)):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
        translated = model.generate(**inputs)
        decoded = [tokenizer.decode(t, skip_special_tokens=True) for t in translated]
        translations.extend(decoded)
    return translations

translations_opus = translate_opus(test_fr)

bleu = sacrebleu.corpus_bleu(translations_opus, [test_en])
print(f"\nOPUS-MT Transformer BLEU score: {bleu.score:.2f}")

100%|██████████| 94/94 [1:03:08<00:00, 40.31s/it]



OPUS-MT Transformer BLEU score: 37.99


In [8]:
# Tests

print("\nSample translations:")
for i in range(5):
    print(f"\nFR:  {test_fr[i]}")
    print(f"REF: {test_en[i]}")
    print(f"OPUS: {translations_opus[i]}")




Sample translations:

FR:  L'affaire NSA souligne l'absence totale de débat sur le renseignement
REF: NSA Affair Emphasizes Complete Lack of Debate on Intelligence
OPUS: The NSA case highlights the total absence of intelligence debate

FR:  Comment expliquer l'attitude contradictoire du gouvernement français, qui d'un coté s'offusque en public en convoquant l'ambassadeur des Etats-Unis le 21 octobre, et de l'autre interdit le survol du territoire par l'avion présidentiel bolivien, sur la base de la rumeur de la présence à son bord d'Edward Snowden ?
REF: Why the contradictory attitude of the French government? On the one hand, it publicly takes offence and summons the Ambassador of the United States on October 21 and, on the other, it forbids the Bolivian president's plane to enter its air space on the basis of a rumor that Edward Snowden was on board?
OPUS: How can we explain the contradictory attitude of the French government, which on the one hand offends in public by summoning the